# UCB-FE Experiments

This notebooks provides all neccessary calculations for the UCB-FE experiments.
Logic about using ML models are implemeted in the [utils file](./utils_ucb_fe.py).

Note that you can either use pretrained models with weights we provided or train all models by your self. The whole training process will take approximately 6 hours on GPU (all models), calculating metrics on CPU will take nearly 1 hour (all models).

P.S. It is possible to verify the code by excluding the tabm model from the calculations, reducing the model training time to 1 hour.

In [ ]:
import numpy  as np
import pandas as pd

In [ ]:
#params
cold_periods = [0, 10, 100, 200, 500]
delta = 1.5
top_n = 10
tail_m = 30

In [ ]:
# Read data
df =  pd.read_parquet('data/results/df_ucb_fe.parquet')

model_names = ['catboost'] #'lightgbm', 'xgboost', 'tabm', 'tabnet'

save_columns = ['target', 'request_id', 'item_id']

for name in model_names:
    save_columns.append('base_ctr_pred_' + name)
    for T in cold_periods:
        save_columns.append('ucb_ctr_pred_T_'+str(T) + '_' + name)

df = df[save_columns]

In [ ]:

import warnings
warnings.filterwarnings("ignore")

class NDCG:
        default_name = "NDCG"

        def __init__(
            self,
            target_column: str,
            discount_base: float,
            top_k = None,
        ):
            self.target_column = target_column
            self.discount_base = discount_base
            self.top_k = top_k

        def __call__(self, serp: pd.DataFrame, rank_column: str) -> float:
            if serp[self.target_column].nunique() == 1:
                return np.nan

            if self.top_k is None:
                top_k = len(serp)
            else:
                top_k = min(len(serp), self.top_k)

            discount_weights = self.discount_base ** np.arange(top_k)
            serp_sorted_by_rank = serp.sort_values(by=rank_column, ascending=False)[:top_k]
            serp_sorted_by_target = serp.sort_values(by=self.target_column, ascending=False)[:top_k]

            score = (discount_weights * serp_sorted_by_rank[self.target_column]).sum()
            norm = (discount_weights * serp_sorted_by_target[self.target_column]).sum()
            return score / (norm + 1e-5)

        @property
        def name(self):
            name = self.default_name
            if self.top_k is not None:
                name += f"@{self.top_k}"
            name += f"_base{self.discount_base}"
            return name

        @property
        def agg_mode(self) -> str:
            return "mean"
        
    

ndcg_score = NDCG(target_column = 'target', discount_base = 0.8)
df_grouped = df.groupby('request_id')
    

ndcg_data = []

columns_ndcg = ['NDCG base']


for T in cold_periods:
    columns_ndcg.append('NDCG fe-ucb, T = ' + str(T))

for name in model_names:
    ndcg = []

    ndcg_old =  df_grouped.apply(ndcg_score, rank_column= 'base_ctr_pred_' + name)

    ndcg.append(ndcg_old.mean())

    for T in cold_periods:
        print("period = ", T)    
        ndcg_new = df_grouped.apply(ndcg_score, rank_column= 'ucb_ctr_pred_T_'+str(T) + '_' + name)
    
        ndcg.append(ndcg_new.mean())
    
    ndcg_data.append(ndcg)

result_ndcg = pd.DataFrame(
    data = ndcg_data, 
    columns= columns_ndcg, 
    index= model_names
)


In [ ]:
result_ndcg.to_csv('data/results/ndcg.csv', index = True)
result_ndcg